# metric modular file 

In [1]:
import numpy as np
from scipy.spatial import ConvexHull

# Empirically verified via inspect_sticky_actions.py (representation='raw'):
#   SPRINT_STICKY_INDEX = 8   (flips 0->1 on SPRINT, persists through IDLE,
#                              flips back on RELEASE_SPRINT)
#   DRIBBLE_STICKY_INDEX = 9  (same test, confirmed alongside sprint)
SPRINT_STICKY_INDEX = 8
DRIBBLE_STICKY_INDEX = 9

# GRF pitch area, normalised coords: x in [-1,1], y in [-0.42,0.42] -> 2.0*0.84
GRF_PITCH_AREA = 1.68

In [2]:
def compute_sap(sticky_sprint_log: np.ndarray) -> float:
    """
    Sprint Action Percentage (SAP).
    sticky_sprint_log: shape (n_steps, n_agents), sprint STICKY BIT (0/1)
        read from obs[agent_i]['sticky_actions'][SPRINT_STICKY_INDEX] --
        NOT the discrete action taken that step.
    """
    sprint_state = np.array(sticky_sprint_log, dtype=float)
    total = sprint_state.size
    if total == 0:
        return 0.0
    active = np.sum(sprint_state == 1.0)
    return float((active / total) * 100.0)

In [3]:
def compute_sbf(sticky_sprint_log: np.ndarray) -> float:
    """
    Sprint Bout Frequency (SBF): avg sprint ON<->OFF transitions per agent.
    Same input shape as compute_sap.
    """
    sprint_state = np.array(sticky_sprint_log, dtype=float)
    if sprint_state.ndim < 2 or sprint_state.shape[0] < 2:
        return 0.0
    n_agents = sprint_state.shape[1]
    if n_agents == 0:
        return 0.0
    total_transitions = 0
    for agent_idx in range(n_agents):
        agent_sprint = sprint_state[:, agent_idx]
        total_transitions += np.sum(np.abs(np.diff(agent_sprint)) > 0)
    return float(total_transitions / n_agents)

In [4]:
def compute_mecha(position_log: np.ndarray, possession_log: np.ndarray) -> float:
    """
    Mean Episode Convex Hull Area (MECHA) -- formation integrity.
    Computed only over possession timesteps (defensive compression is
    tactically correct play and should not be penalised).
    position_log: (n_steps, n_agents, 2). possession_log: (n_steps,) bool.
    """
    positions = np.array(position_log)
    possession = np.array(possession_log, dtype=bool)

    assert len(positions) == len(possession), (
        f"Length mismatch: {len(positions)} positions vs "
        f"{len(possession)} possession flags -- fix upstream logging."
    )

    if not np.any(possession) or len(positions) == 0:
        return 0.0

    possession_positions = positions[possession]
    hull_areas = []
    for step_pos in possession_positions:
        unique_pos = np.unique(step_pos, axis=0)
        if len(unique_pos) >= 3:
            try:
                hull = ConvexHull(unique_pos)
                hull_areas.append(hull.volume)  # 2D: volume == area
            except Exception:
                pass  # degenerate/near-collinear -- skip this step

    if not hull_areas:
        return 0.0

    mecha = float(np.mean(hull_areas) / GRF_PITCH_AREA)
    return float(np.clip(mecha, 0.0, 1.0))

In [5]:
def compute_cv(position_log: np.ndarray) -> float:
    """
    Centroid Variance (CV) -- sum of per-axis variances of the team centroid
    (trace of the covariance matrix). Summing x and y variance separately
    avoids x dominating due to the pitch being ~2x longer than it is wide.
    """
    positions = np.array(position_log)
    if len(positions) == 0:
        return 0.0
    centroids = np.mean(positions, axis=1)  # (n_steps, 2)
    var_x = np.var(centroids[:, 0])
    var_y = np.var(centroids[:, 1])
    return float(var_x + var_y)

In [6]:
def extract_sprint_sticky_bits(raw_obs_list):
    """Sprint sticky bit per controlled agent, this timestep."""
    return np.array([
        int(agent_obs['sticky_actions'][SPRINT_STICKY_INDEX])
        for agent_obs in raw_obs_list
    ])


def extract_team_positions(raw_obs_list, n_players=5):
    """
    All left-team (our) player positions this timestep, from obs[0]['left_team']
    -- this is team-wide, independent of how many agents you personally control.
    VERIFY n_players slice matches your squad size for 5v5 (print
    raw_obs_list[0]['left_team'].shape and raw_obs_list[0]['left_team_roles']
    once before trusting this at scale).
    """
    left_team = np.array(raw_obs_list[0]['left_team'])
    return left_team[:n_players]


def extract_possession(raw_obs_list):
    """
    ball_owned_team: -1 = none, 0 = left (us), 1 = right.
    This is an explicit numeric field in raw -- no one-hot guessing needed.
    """
    owned_team = raw_obs_list[0]['ball_owned_team']
    return bool(owned_team == 0)

In [7]:
def _test_metrics():
    # SAP: all sprint -> 100%, none -> 0%, half -> ~50%
    assert abs(compute_sap(np.ones((100, 4))) - 100.0) < 0.01
    assert abs(compute_sap(np.zeros((100, 4))) - 0.0) < 0.01
    alt = np.array([[1 if i % 2 == 0 else 0] * 4 for i in range(100)])
    assert abs(compute_sap(alt) - 50.0) < 1.0

    # SBF: constant -> 0 transitions, alternating -> near max
    assert compute_sbf(np.ones((100, 4))) == 0.0
    assert compute_sbf(alt) > 50

    # MECHA: spread > clustered; no possession -> 0
    spread = [np.array([[-0.5,-0.2],[0.5,-0.2],[-0.5,0.2],[0.5,0.2],[0.0,0.0]])] * 50
    clustered = [np.array([[0.01*i,0.01*j] for i,j in [(0,0),(1,0),(0,1),(1,1),(0,2)]])] * 50
    poss = [True] * 50
    assert compute_mecha(spread, poss) > compute_mecha(clustered, poss)
    assert compute_mecha(spread, [False]*50) == 0.0

    # CV: stationary -> ~0, moving -> > 0
    static = [np.array([[0.1,0.1],[0.2,0.1],[0.3,0.1],[0.4,0.1],[0.5,0.1]])] * 100
    assert compute_cv(static) < 0.001

    print("All metric self-tests passed.")

_test_metrics()

All metric self-tests passed.
